# Incremental pipeline vs full reclustering

Replay-эксперимент имитирует поступление публикаций по времени. Начальный период кластеризуется full pipeline, затем новые публикации добавляются incremental pipeline. В каждой контрольной точке выполняется full reclustering на том же доступном временном срезе.

In [ ]:
from pathlib import Path
import sys

import matplotlib.pyplot as plt
import pandas as pd

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == 'notebooks':
    PROJECT_ROOT = PROJECT_ROOT.parent
for path in [PROJECT_ROOT, PROJECT_ROOT / 'src']:
    if str(path) not in sys.path:
        sys.path.insert(0, str(path))

RESULT_DIR = PROJECT_ROOT / 'data/artifacts/incremental_pipeline_benchmark_components'

## Запуск

Следующая команда использует candidate pool и готовый id-aware cache embeddings. Параметры `bootstrap-days` и `checkpoint-days` можно менять для проверки других режимов.

In [ ]:
import subprocess

subprocess.run([
    sys.executable,
    str(PROJECT_ROOT / 'scripts/benchmark_incremental_pipeline.py'),
    '--project-root', str(PROJECT_ROOT),
    '--bootstrap-days', '14',
    '--checkpoint-days', '7',
    '--output-dir', str(RESULT_DIR),
], check=True)

In [ ]:
metrics = pd.read_csv(RESULT_DIR / 'incremental_vs_full_metrics.csv', parse_dates=['checkpoint_date'])
display(metrics[[
    'checkpoint_date', 'days_since_bootstrap', 'rows', 'pairwise_precision',
    'pairwise_recall', 'pairwise_f1', 'fragmented_full_cluster_rate',
    'false_merged_incremental_cluster_rate', 'novelty_label_agreement',
    'p_significant_mae', 'created_clusters_in_batch',
]])

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4))
metrics.plot(x='days_since_bootstrap', y=['pairwise_precision', 'pairwise_recall', 'pairwise_f1'], marker='o', ylim=(0, 1.02), ax=axes[0])
metrics.plot(x='days_since_bootstrap', y=['fragmented_full_cluster_rate', 'false_merged_incremental_cluster_rate'], marker='o', ylim=(0, 1.02), ax=axes[1])
metrics.plot(x='days_since_bootstrap', y=['novelty_label_agreement', 'significance_binary_agreement'], marker='o', ylim=(0, 1.02), ax=axes[2])
axes[0].set_title('Cluster pair metrics')
axes[1].set_title('Cluster structural errors')
axes[2].set_title('Novelty agreement')
plt.tight_layout();

In [ ]:
threshold_rows = []
for threshold in [0.95, 0.90, 0.85, 0.80]:
    crossed = metrics.loc[metrics['pairwise_f1'] < threshold]
    threshold_rows.append({
        'pairwise_f1_threshold': threshold,
        'first_crossing_days': None if crossed.empty else int(crossed.iloc[0]['days_since_bootstrap']),
        'first_crossing_date': None if crossed.empty else crossed.iloc[0]['checkpoint_date'],
    })
display(pd.DataFrame(threshold_rows))

In [ ]:
assignments = pd.read_csv(RESULT_DIR / 'incremental_vs_full_assignments.csv')
last_checkpoint = assignments['checkpoint'].max()
last_assignments = assignments[assignments['checkpoint'].eq(last_checkpoint)]
fragmented = (
    last_assignments.groupby('cluster_id_full')['cluster_id_incremental']
    .nunique().sort_values(ascending=False).rename('incremental_cluster_count')
)
display(fragmented.head(20).to_frame())

In [ ]:
component_columns = [
    column for column in [
        'baseline_component_id_full', 'baseline_component_id_incremental',
        'assignment_method_full', 'assignment_method_incremental',
    ] if column in last_assignments.columns
]
display(last_assignments[['news_id', 'cluster_id_full', 'cluster_id_incremental', *component_columns]].head(20))

## Интерпретация

- Падение pairwise recall при высокой precision означает накопление фрагментации: incremental создаёт отдельные кластеры, которые full pipeline позднее объединяет.
- Падение precision означает ложные объединения incremental pipeline.
- Частоту batch-рекластеризации следует выбирать по первой контрольной точке, где метрика пересекает допустимый продуктовый порог.